# 03. Tinh chỉnh siêu tham số
Nhánh GitHub: `03-tuning`

Chỉ dùng validation (fold 1), không dùng fold test. E1, E5: learning rate (3e-4, 1e-3) × batch size (16, 32, 64). E3, E7: mức che SpecAugment. Kết quả lưu ở `tuning_*.csv`; giá trị tốt nhất (nếu dùng) sửa vào `CFG` trong `esc_lib.py`. Learning rate riêng của E1–E8 được chọn trong notebook của từng thí nghiệm.

In [ ]:
!pip -q install soundfile
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/ESC50')
from esc_lib import *
setup()

Mounted at /content/drive
GPU: Tesla T4 | VRAM 15.6 GB | torch 2.14.1+cu130 | CUDA 13.0
RAM trống 11.8 GB | ổ đĩa trống 120.8 GB
Mixup: BẬT | quick: False | kết quả lưu ở: /content/drive/MyDrive/ESC50/results_mixup


In [ ]:
meta, X, X_DEV, y, folds = load_data()

  0%|          | 0/2000 [00:00<?, ?it/s]

X: (2000, 110250) | y: (2000,) | nằm trên: cuda:0


In [ ]:
GRID_LR_BATCH = [dict(LR=lr, BATCH_SIZE=b) for lr in LR_GRID for b in (16, 32, 64)]
GRID_MASK = [dict(FREQ_MASK_RATIO=r, TIME_MASK_PARAM=t) for r, t in ((0.08, 10), (0.125, 20), (0.2, 30))]

### Learning rate × batch size (E1, E5)

In [ ]:
for e in ("E1", "E5"):
    print(f"\n=== {e}: learning rate và batch size ===")
    res = tune(e, GRID_LR_BATCH)
    res.to_csv(RESULT_DIR / f"tuning_lr_batch_{e}.csv", index=False)
    display(res)


=== E1: learning rate và batch size ===


train:   0%|          | 0/80 [00:00<?, ?it/s]

E1 {'LR': 0.0003, 'BATCH_SIZE': 16} -> 0.8371


train:   0%|          | 0/80 [00:00<?, ?it/s]

E1 {'LR': 0.0003, 'BATCH_SIZE': 32} -> 0.8254


train:   0%|          | 0/80 [00:00<?, ?it/s]

... (18 outputs left out)

E5 {'LR': 0.001, 'BATCH_SIZE': 32} -> 0.7785


train:   0%|          | 0/80 [00:00<?, ?it/s]

E5 {'LR': 0.001, 'BATCH_SIZE': 64} -> 0.7734


,LR,BATCH_SIZE,val_macro_f1,best_epoch
1,0.0003,32,0.779377,76
4,0.0010,32,0.778547,60
0,0.0003,16,0.776048,60
5,0.0010,64,0.773366,75
3,0.0010,16,0.772961,73
2,0.0003,64,0.771801,59


### Mức che SpecAugment (E3, E7)

In [ ]:
for e in ("E3", "E7"):
    print(f"\n=== {e}: mức che SpecAugment ===")
    res = tune(e, GRID_MASK)
    res.to_csv(RESULT_DIR / f"tuning_mask_{e}.csv", index=False)
    display(res)


=== E3: mức che SpecAugment ===


train:   0%|          | 0/80 [00:00<?, ?it/s]

E3 {'FREQ_MASK_RATIO': 0.08, 'TIME_MASK_PARAM': 10} -> 0.8581


train:   0%|          | 0/80 [00:00<?, ?it/s]

E3 {'FREQ_MASK_RATIO': 0.125, 'TIME_MASK_PARAM': 20} -> 0.8518


train:   0%|          | 0/80 [00:00<?, ?it/s]

... (6 outputs left out)

E7 {'FREQ_MASK_RATIO': 0.125, 'TIME_MASK_PARAM': 20} -> 0.8023


train:   0%|          | 0/80 [00:00<?, ?it/s]

E7 {'FREQ_MASK_RATIO': 0.2, 'TIME_MASK_PARAM': 30} -> 0.8072


,FREQ_MASK_RATIO,TIME_MASK_PARAM,val_macro_f1,best_epoch
2,0.200,30,0.807209,72
0,0.080,10,0.804614,56
1,0.125,20,0.802321,80
